In [ ]:
import os, sys
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
import pandas as pd
import functions1 as f1  
import functions2 as f2
import importlib
import config
import risk_matrix
import books

importlib.reload(books)
importlib.reload(risk_matrix)

window_start="2025-1-02"
holdings=books.IBKR_live

rets_df, prices_df, weights =risk_matrix.build_returns_matrix_in_chf(
                holdings,
                params=config.params,
                window_start=window_start)
results = risk_matrix.portfolio_risk(rets_df, weights)
pf_vol = results["port_vol"]
summary = results["summary"]
# print('summary', summary)
assets = summary.index.values
assets = [a for a in assets if 'CASH' not in a]

weights = summary["Weight"].astype(float)
mrc = summary["MRC"].astype(float)
desired_dd = 0.04

df = f2.base_ccy_assets_px_df(holdings, config.params)

# ohlc_by_asset: dict like {asset: df_with_cols['High','Low','Close']}
# If your variable is named differently, replace 'ohlc_by_asset' below.
series_high, series_low, series_close = [], [], []
for h in holdings:
    # if h.get('type') == 'cash':
    #     continue
    # df_a = ohlc_by_asset[a]
    t = h.get('ticker', None)
    name = h.get('name', None)
    # print(f"Fetching OHLC for {h}...")
    s = f1.fetch_csv_robust(t,params=config.params)
    series_high.append(s['High'].rename(name))
    series_low.append(s['Low'].rename(name))
    series_close.append(s['Close'].rename(name))

high_df  = pd.concat(series_high,  axis=1)   # columns = assets, index auto-aligned
low_df   = pd.concat(series_low,   axis=1)
close_df = pd.concat(series_close, axis=1)

# ATR% (vectorized, Wilder)
prev_close = close_df.shift(1)
tr = (high_df - low_df) \
    .combine((high_df - prev_close).abs(), np.maximum) \
    .combine((low_df  - prev_close).abs(), np.maximum)
atr = tr.ewm(alpha=1/14, adjust=False).mean()
atr_pct = (atr.iloc[-1] / close_df.iloc[-1]).astype(float).clip(lower=1e-6)

# Ensure 'w' exists (you used it below)
w = weights
# 1) Compute ATR% (Wilder) from OHLC in the SAME currency basis as weights/returns (CHF)
# Expect DataFrames: high_df, low_df, close_df with columns matching 'assets'
# If you have them under different names, adjust below.
high = high_df[assets]
low = low_df[assets]
close = close_df[assets]

prev_close = close.shift(1)
tr = pd.concat([
    (high - low),
    (high - prev_close).abs(),
    (low - prev_close).abs()
], axis=1).groupby(level=0, axis=1).max()  # max of the three TR components per asset

atr = tr.ewm(alpha=1/14, adjust=False).mean()       # Wilder's 14-period RMA
atr_pct = (atr.iloc[-1] / close.iloc[-1]).astype(float).clip(lower=1e-6)

# 2) Per-asset DD budgets by absolute TRC (or use equal share)
trc_abs = (w * mrc).abs()
share = trc_abs / trc_abs.sum()
b = desired_dd * share                                # sum(b) = desired_dd

# 3) L1 stop distances and ATR multiples
abs_w = w.abs().clip(lower=1e-12)
stop_pct = (b / abs_w).clip(0.002, 0.10)              # optional floors/ceilings on distance (0.2%–10%)
k_multiple = (stop_pct / atr_pct).clip(0.25, 6.0)     # optional caps on ATR multiples

# 4) Stop prices
last_px = close.iloc[-1].astype(float).reindex(assets)
stop_px = pd.Series(
    np.where(w >= 0, last_px * (1.0 - stop_pct), last_px * (1.0 + stop_pct)),
    index=assets, name="stop_price"
)

# 5) Sanity check: worst-case sum should hit target DD
dd_check = float((abs_w * stop_pct).sum())
print(f"L1 sum check ~ {dd_check:.4f} (target {desired_dd:.4f})")

out = pd.DataFrame({
    "Weight": w,
    "ATR_pct": atr_pct,
    "stop_pct": stop_pct,
    "k_ATR_mult": k_multiple,
    "last_px": last_px,
    "stop_px": stop_px
}).round(6)
print(out)

After alignment only 193 rows remain (expected 286 days 00:00:00). Data source may not have full history.
LOOKBACK DAYS/REGIME: 2025-01-02 00:00:00 to 2025-10-15 00:00:00  (286 days)
Total portfolio value (CHF): 34738.47
Cache file /Users/alexwebb/laptop_coding/risk_matrix/cache/.csv does not exist
 - downloading fresh data


KeyboardInterrupt: 

In [ ]:



trc = (weights * mrc).abs()
share = trc / trc.sum()  
# print('share', share)        # risk share by asset
desired_dd = 0.04
b = desired_dd * share 
# print('b',b)    

# STOP TERMS
abs_weigths = np.abs(weights) + 1e-18
stop_pct_L1 = b / abs_weigths

# BLEND AND ENFORCE L1 CAP
lam = 0.3
stop_pct = (1 - lam) * 

